# E-16: exact examples (D type, H16)

Registration: `doc/2026-10-07_experiment_registration.md` (parent repository), E-16 and Amendment 2. Each example is solved exactly with sympy and compared with floating-point values at absolute tolerance $10^{-10}$. An example that does not agree is a negative result and does not enter the manuscript.

Run only through `python3 notebooks/experiments/run_registered.py E-16 stage0 14_E16_exact_examples.ipynb`. A solve that returns a non-optimal status is a negative result for its example. A solver that raises stops the notebook, and the run is recorded as failed (registration §1.1: no `try`-based handling).

In [1]:
import warnings
from fractions import Fraction

import numpy as np
import sympy as sp

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
rec = outputs.RunRecorder(16, "stage0")

TOL = 1e-10
CLARABEL = {"tol_gap_abs": 1e-12, "tol_gap_rel": 1e-12, "tol_feas": 1e-12}  # Amendment 2
SPURIOUS_MATMUL = r"(?:divide by zero|overflow|invalid value) encountered in matmul$"
checks = []  # (example, quantity, exact sympy value, float value)


def check(example, quantity, exact, value):
    """A non-finite value (a failed or non-optimal solve) is a disagreement, not an error."""
    checks.append((example, quantity, sp.nsimplify(exact), float(value)))


NAN = float("nan")

## Exact values (sympy)

In [2]:
R = sp.Rational
b, a = sp.symbols("b a", real=True)

# (i) PR4a 2(a-4): n = p = 1, phi = 1, b_m = 3, g(a) = a^2/2, lambda = 1/2, B = [-1, 1].
P_i = b**2 / 2 - 3 * b + sp.Abs(b) / 2
cand = {x: P_i.subs(b, x) for x in (-1, 0, 1)}
assert all(sp.diff(b**2 / 2 - 3 * b + b / 2, b).subs(b, x) < 0 for x in (R(1, 2), 1))
beta_i = min(cand, key=cand.get)
exact_i = {"beta": sp.Integer(beta_i), "alpha": sp.Integer(beta_i), "imbalance": sp.Integer(beta_i) - 3,
           "weight_program": R(5, 2), "weight_program_value": R(25, 8)}

# (ii) PR4a §4(a): BP(omega=1, C=0), g(a) = a^2 - 2a, X = (1, -1), phi = x, b_m = 3/2, lambda = 0.
Q = b**2 / 4 + 1 - R(3, 2) * b
assert sp.solve(sp.diff(Q, b), b) == [3]  # the only stationary point lies outside (-2, 2)
inf_Q = sp.limit(Q, b, 2, dir="-")
a1, a2 = sp.symbols("a1 a2", nonnegative=True)
obj_w = (a1**2 - 2 * a1 + a2**2 - 2 * a2) / 2
on_line = sp.expand(obj_w.subs(a1, a2 + 3))
assert sp.diff(on_line, a2).subs(a2, 0) > 0 and sp.diff(on_line, a2, 2) > 0
exact_ii = {"inf_not_attained": inf_Q, "w1": sp.Integer(3), "w2": sp.Integer(0), "value": obj_w.subs({a1: 3, a2: 0})}

Z = (-1, 0, 1)


def E(e0, f):
    return R(1, 3) * sum(f(e, z) for e, z in zip(e0, Z))


def basis_z(deg):
    return [lambda z, k=k: sp.Integer(z) ** k for k in range(deg + 1)]


def balance(e0, alpha_t, deg):
    # treated-arm population balance E[e0 alpha phi_k] = E[phi_k], k <= deg (the first-order conditions)
    return [E(e0, lambda e, z, f=f: e * alpha_t(z) * f(z)) - E(e0, lambda e, z, f=f: f(z)) for f in basis_z(deg)]


def sq_treated(e0, deg):
    """SQ (linear link): the strictly convex criterion's unique minimizer solves the linear balance equations."""
    cs = sp.symbols(f"c0:{deg + 1}")
    alpha_t = lambda z: sum(c * sp.Integer(z) ** k for k, c in enumerate(cs))  # noqa: E731
    sol = sp.solve(balance(e0, alpha_t, deg), cs, dict=True)
    assert len(sol) == 1
    w = [sp.nsimplify(alpha_t(z).subs(sol[0])) for z in Z]
    est = E(e0, lambda e, z: e * alpha_t(z).subs(sol[0]) * z**2)
    return w, sp.nsimplify(est)


def ukl_treated_true(e0, deg):
    """UKL(C=1): if log(1/e0 - 1) is a polynomial of degree <= deg on the support, alpha0 is in the model;
    alpha0 satisfies the balance equations exactly, and strict convexity makes it the unique minimizer."""
    cs = sp.symbols(f"c0:{deg + 1}")
    eqs = [sum(c * sp.Integer(z) ** k for k, c in enumerate(cs)) - sp.log(1 / e - 1) for e, z in zip(e0, Z)]
    sol = sp.solve(eqs, cs, dict=True)
    assert len(sol) == 1, "dual coordinate of alpha0 not in the span"
    alpha_t = lambda z: 1 + sp.exp(sum(c * sp.Integer(z) ** k for k, c in enumerate(cs)).subs(sol[0]))  # noqa: E731
    assert all(sp.simplify(alpha_t(z) - 1 / e) == 0 for e, z in zip(e0, Z))
    assert all(sp.simplify(r) == 0 for r in balance(e0, alpha_t, deg))
    est = E(e0, lambda e, z: e * alpha_t(z) * z**2)
    return [sp.nsimplify(sp.simplify(alpha_t(z))) for z in Z], sp.nsimplify(sp.simplify(est))


theta0 = R(2, 3)
e_iii = [R(1, 4), R(1, 2), R(3, 4)]
w_sq, est_sq = sq_treated(e_iii, 1)
exact_iii = {
    "sq_w": w_sq, "sq_estimand": est_sq, "sq_bias": est_sq - theta0,
    "ukl_bias": ukl_treated_true(e_iii, 1)[1] - theta0,
    "sq_bias_z2": sq_treated(e_iii, 2)[1] - theta0,
    "ukl_bias_z2": ukl_treated_true(e_iii, 2)[1] - theta0,
}

e_iv = [R(6, 17), R(3, 17), R(2, 17)]
w_ukl = [1 + sp.Integer(2) ** (z + 2) for z in Z]
assert E(e_iv, lambda e, z: e * (1 + 2 ** (z + 2))) == 1 and E(e_iv, lambda e, z: e * (1 + 2 ** (z + 2)) * z) == 0
est_ukl = E(e_iv, lambda e, z: e * (1 + 2 ** (z + 2)) * z**2)
w_sq_iv, est_sq_iv = sq_treated(e_iv, 1)
exact_iv = {"ukl_w": w_ukl, "ukl_estimand": est_ukl, "ukl_bias": est_ukl - theta0, "sq_bias": est_sq_iv - theta0}

# (v) PR4b active-constraint counterexample (Amendment 2): exact constrained minimizer by KKT.
xs = (0, 1, 2)
alpha0 = [R(3, 4), R(3, 4), R(3, 2)]
b1, b2 = sp.symbols("b1 b2", real=True)
f_v = sum((b1 + b2 * x) ** 2 for x in xs) / 3 - R(1, 2) * (4 * b1 + 5 * b2)
H = sp.hessian(f_v, (b1, b2))
assert all(m > 0 for m in (H[0, 0], H.det()))  # positive definite: strictly convex
beta_v = (R(3, 4), R(3, 10))
at = {b1: beta_v[0], b2: beta_v[1]}
cons = [(b1 + b2 * x) - R(3, 4) for x in xs] + [R(3, 2) - (b1 + b2 * x) for x in xs]  # each >= 0
values = [c.subs(at) for c in cons]
assert all(v >= 0 for v in values)
active = [c for c, v in zip(cons, values) if v == 0]
grad_f = sp.Matrix([sp.diff(f_v, v) for v in (b1, b2)]).subs(at)
mus = sp.symbols(f"mu0:{len(active)}", nonnegative=True)
kkt = grad_f - sum((mu * sp.Matrix([sp.diff(c, v) for v in (b1, b2)]) for mu, c in zip(mus, active)), sp.zeros(2, 1))
sol = sp.solve(list(kkt), mus, dict=True)
assert sol and all(val >= 0 for val in sol[0].values())  # KKT with non-negative multipliers
a_v = [beta_v[0] + beta_v[1] * x for x in xs]
e_v = [ai - a0 for ai, a0 in zip(a_v, alpha0)]
exact_v = {"beta1": beta_v[0], "beta2": beta_v[1], "alpha": a_v, "norm_sq": sum(ei**2 for ei in e_v) / 3,
           "mean_e": sum(e_v) / 3}
assert exact_v["norm_sq"] > R(1, 32)
exact_v

{'beta1': 3/4,
 'beta2': 3/10,
 'alpha': [3/4, 21/20, 27/20],
 'norm_sq': 3/80,
 'mean_e': 1/20}

## Floating-point values

In [3]:
def rows_from(probs):
    den = int(np.lcm.reduce([int(Fraction(str(p)).denominator) for p in probs.values()]))
    rows = []
    for (d, z), p in probs.items():
        rows += [(d, z)] * int(Fraction(str(p)) * den)
    return np.array(rows, float)


def ate_rows(e0):
    probs = {}
    for z, e in zip(Z, e0):
        probs[(1, z)] = R(1, 3) * e
        probs[(0, z)] = R(1, 3) * (1 - e)
    return rows_from(probs)


ones = gr.CallableBasis(lambda X: np.ones((np.atleast_2d(X).shape[0], 1)))
ident = gr.CallableBasis(lambda X: np.atleast_2d(X)[:, :1].astype(float))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    warnings.filterwarnings("ignore", message=SPURIOUS_MATMUL, category=RuntimeWarning)

    # (i) genriesz SQ is g = a^2 (alpha = v/2); the problem with g = a^2/2 is the same with beta = 2 beta_tilde,
    # so lambda = 1/2 and the l1 radius 2.
    f_i = gr.CallableFunctional(lambda x, gam: 3.0 * gam(x))
    m_i = gr.GRRGLM(basis=ones, generator=gr.SquaredGenerator(C=0.0), functional=f_i,
                    penalty="l1", lam=0.5, l1_radius=2.0)
    r_i = m_i.fit(np.zeros((1, 1)))
    ok_i = r_i.status == "ok"
    al_i = m_i.predict_alpha(np.zeros((1, 1)))[0] if ok_i else NAN
    check("i", "beta", exact_i["beta"], m_i.beta_[0] / 2 if ok_i else NAN)
    check("i", "alpha", exact_i["alpha"], al_i)
    check("i", "imbalance", exact_i["imbalance"], al_i - 3.0)
    w_i = gr.weight_program_certificate(X=np.zeros((1, 1)), basis=ones, functional=f_i,
                                        generator=gr.SquaredGenerator(C=0.0), lam=0.5,
                                        backend="cvxpy", solver_options=CLARABEL)
    opt_i = w_i.solver_status == "optimal"
    check("i", "weight_program", exact_i["weight_program"], w_i.alpha[0] if opt_i else NAN)
    # the certificate's SQ minimizes a^2; the registered program minimizes a^2/2 (Amendment 2)
    check("i", "weight_program_value", exact_i["weight_program_value"], w_i.primal_value / 2 if opt_i else NAN)

    # (ii)
    f_ii = gr.CallableFunctional(lambda x, gam: (3.0 if x[0] > 0 else 0.0) * gam(x))
    bp = gr.BPGenerator(omega=1.0, C=0.0, branch_fn=lambda x: 1)
    X_ii = np.array([[1.0], [-1.0]])
    r_ii = gr.GRRGLM(basis=ident, generator=bp, functional=f_ii, penalty=None, lam=0.0).fit(X_ii)
    grr_status_ii = r_ii.status  # recorded; the exact non-existence is the sympy statement above
    w_ii = gr.weight_program_certificate(X=X_ii, basis=ident, functional=f_ii, generator=bp, lam=0.0,
                                         backend="cvxpy", solver_options=CLARABEL)
    opt_ii = w_ii.solver_status == "optimal"
    check("ii", "w1", exact_ii["w1"], w_ii.alpha[0] if opt_ii else NAN)
    check("ii", "w2", exact_ii["w2"], w_ii.alpha[1] if opt_ii else NAN)
    check("ii", "value", exact_ii["value"], w_ii.primal_value if opt_ii else NAN)

    # (iii), (iv): the finite-support population as integer row multiplicities.
    sign = lambda X: np.where(np.atleast_2d(X)[:, 0] == 1, 1, -1)  # noqa: E731

    def poly(deg):
        return gr.CallableBasis(lambda Zc: np.column_stack([np.atleast_2d(Zc)[:, 0] ** k for k in range(deg + 1)]))

    def fit(X, gen, deg, offset=None):
        m = gr.GRRGLM(basis=gr.TreatmentInteractionBasis(base_basis=poly(deg)), generator=gen,
                      functional=gr.ATEFunctional(0), penalty=None, lam=0.0, offset=offset)
        r = m.fit(X)
        if r.status != "ok":
            return NAN, [NAN, NAN, NAN]
        a_hat = m.predict_alpha(X)
        est = float(np.mean(a_hat * np.where(X[:, 0] == 1, X[:, 1] ** 2, 0.0)))
        w = [m.predict_alpha(np.array([[1.0, z]]))[0] for z in Z]
        return est, w

    ukl = gr.UKLGenerator(C=1.0, branch_fn=sign)
    off = gr.offset_from_alpha(ukl, lambda X: np.where(X[:, 0] == 1, 2.0, -2.0))
    X_iii = ate_rows(e_iii)
    est, w = fit(X_iii, gr.SquaredGenerator(C=0.0), 1)
    for z, wz, ez in zip(Z, w, exact_iii["sq_w"]):
        check("iii", f"sq_w(z={z})", ez, wz)
    check("iii", "sq_estimand", exact_iii["sq_estimand"], est)
    check("iii", "sq_bias", exact_iii["sq_bias"], est - 2 / 3)
    check("iii", "ukl_bias", exact_iii["ukl_bias"], fit(X_iii, ukl, 1, off)[0] - 2 / 3)
    check("iii", "sq_bias_z2", exact_iii["sq_bias_z2"], fit(X_iii, gr.SquaredGenerator(C=0.0), 2)[0] - 2 / 3)
    check("iii", "ukl_bias_z2", exact_iii["ukl_bias_z2"], fit(X_iii, ukl, 2, off)[0] - 2 / 3)

    X_iv = ate_rows(e_iv)
    est, w = fit(X_iv, ukl, 1, off)
    for z, wz, ez in zip(Z, w, exact_iv["ukl_w"]):
        check("iv", f"ukl_w(z={z})", ez, wz)
    check("iv", "ukl_estimand", exact_iv["ukl_estimand"], est)
    check("iv", "ukl_bias", exact_iv["ukl_bias"], est - 2 / 3)
    check("iv", "sq_bias", exact_iv["sq_bias"], fit(X_iv, gr.SquaredGenerator(C=0.0), 1)[0] - 2 / 3)

    # (v) Amendment 2: population SQ-Riesz criterion E[alpha^2] - 2 m(alpha) on the range-constrained model.
    import cvxpy as cp

    bv = cp.Variable(2)
    av = [bv[0] + bv[1] * x for x in xs]
    crit = sum(ai**2 for ai in av) / 3 - 2 * (av[0] + av[1] + 2 * av[2]) / 4
    cons = [ai >= 0.75 for ai in av] + [ai <= 1.5 for ai in av]
    prob = cp.Problem(cp.Minimize(crit), cons)
    prob.solve(solver="CLARABEL", **CLARABEL)
    b_hat = np.asarray(bv.value, float) if prob.status == "optimal" else np.array([NAN, NAN])
    a_hat = np.array([b_hat[0] + b_hat[1] * x for x in xs])
    e_hat = a_hat - np.array([0.75, 0.75, 1.5])
    check("v", "beta1", exact_v["beta1"], b_hat[0])
    check("v", "beta2", exact_v["beta2"], b_hat[1])
    for x, ah, ax in zip(xs, a_hat, exact_v["alpha"]):
        check("v", f"alpha(x={x})", ax, ah)
    check("v", "norm_sq", exact_v["norm_sq"], np.mean(e_hat**2))
    check("v", "mean_e", exact_v["mean_e"], np.mean(e_hat))

warning_messages = [str(w.message) for w in caught]
len(checks), warning_messages

(29,
 ['Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)',
  'Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)',
  'Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)',
  'Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)',
  'Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this ope

## Judgement and outputs

In [4]:
import pandas as pd

rows = [{"kind": "check", "example": ex, "quantity": q, "exact": str(v), "exact_latex": sp.latex(v),
         "float": f, "abs_diff": abs(f - float(v))} for ex, q, v, f in checks]
rows.append({"kind": "exact_only", "example": "ii", "quantity": "inf_not_attained",
             "exact": str(exact_ii["inf_not_attained"]), "exact_latex": sp.latex(exact_ii["inf_not_attained"]),
             "float": NAN, "abs_diff": NAN})
table = pd.DataFrame(rows)
table["agrees"] = (table["kind"] == "check") & np.isfinite(table["float"]) & (table["abs_diff"] <= TOL)
print(table.to_string(index=False))
rec.write_summary(table)
rec.write_stage0(
    "e16_checks",
    {
        "tolerance": TOL,
        "clarabel_settings": CLARABEL,
        "checks": [
            {k: (None if isinstance(v, float) and not np.isfinite(v) else v) for k, v in r.items()}
            for r in table.to_dict(orient="records")
        ],
        "grr_status_example_ii": grr_status_ii,
        "solver_status": {"i_weight_program": w_i.solver_status, "ii_weight_program": w_ii.solver_status,
                          "v": prob.status},
        "warnings": warning_messages,
    },
)

      kind example             quantity exact    exact_latex         float     abs_diff  agrees
     check       i                 beta     1              1  1.000000e+00 0.000000e+00    True
     check       i                alpha     1              1  1.000000e+00 0.000000e+00    True
     check       i            imbalance    -2             -2 -2.000000e+00 0.000000e+00    True
     check       i       weight_program   5/2    \frac{5}{2}  2.500000e+00 1.376677e-14    True
     check       i weight_program_value  25/8   \frac{25}{8}  3.125000e+00 3.463896e-14    True
     check      ii                   w1     3              3  3.000000e+00 1.332268e-14    True
     check      ii                   w2     0              0  1.422084e-14 1.422084e-14    True
     check      ii                value   3/2    \frac{3}{2}  1.500000e+00 1.243450e-14    True
     check     iii           sq_w(z=-1)  18/5   \frac{18}{5}  3.600000e+00 1.332268e-15    True
     check     iii            sq_w(z=0) 

PosixPath('/Users/mkato/Dropbox/研究/generalizedrieszregression/genriesz/notebooks/experiments/results/E-16/stage0/stage0_e16_checks.json')

In [5]:
# Tables and macros are made from summary.csv only (registration §4).
summ = outputs.read_summary(rec)
checks_ = summ[summ["kind"] == "check"]
verdict = checks_.groupby("example")["agrees"].all().to_dict()


def tx(example, quantity):
    row = summ[(summ["example"] == example) & (summ["quantity"] == quantity)]
    assert len(row) == 1
    return row["exact_latex"].iloc[0]


TITLES = {
    "i": "Coefficient restriction active",
    "ii": "Basu power generator without a minimizer",
    "iii": "Same regressors, different bias",
    "iv": "Reverse direction",
    "v": "Active range constraint",
}
VALUES = {
    "i": rf"$\widehat\beta={tx('i', 'beta')}$, imbalance ${tx('i', 'imbalance')}$, weight program ${tx('i', 'weight_program')}$",
    "ii": rf"infimum ${tx('ii', 'inf_not_attained')}$ not attained; weight program $({tx('ii', 'w1')},{tx('ii', 'w2')})$",
    "iii": rf"SQ estimand ${tx('iii', 'sq_estimand')}$, bias ${tx('iii', 'sq_bias')}$; UKL bias ${tx('iii', 'ukl_bias')}$",
    "iv": (rf"UKL weights $({tx('iv', 'ukl_w(z=-1)')},{tx('iv', 'ukl_w(z=0)')},{tx('iv', 'ukl_w(z=1)')})$, "
           rf"bias ${tx('iv', 'ukl_bias')}$; SQ bias ${tx('iv', 'sq_bias')}$"),
    "v": (rf"$\alpha^*=({tx('v', 'alpha(x=0)')},{tx('v', 'alpha(x=1)')},{tx('v', 'alpha(x=2)')})$, "
          rf"$\|e^*\|^2={tx('v', 'norm_sq')}$, $\mathbb{{E}} e^*={tx('v', 'mean_e')}$"),
}
lines = [r"\begin{tabular}{lll}", r"\hline", r"Example & Exact values & Maximum absolute deviation \\", r"\hline"]
for ex in ("i", "ii", "iii", "iv", "v"):
    if verdict[ex]:
        dev = checks_.loc[checks_["example"] == ex, "abs_diff"].max()
        lines.append(rf"{TITLES[ex]} & {VALUES[ex]} & ${dev:.1e}$ \\")
lines += [r"\hline", r"\end{tabular}"]
rec.write_table("tab_E16", "\n".join(lines) + "\n")

def sx(example, quantity):
    """The exact value in the inline form the manuscript uses (``3/5``, ``-1/15``)."""
    row = summ[(summ["example"] == example) & (summ["quantity"] == quantity)]
    assert len(row) == 1
    return str(row["exact"].iloc[0])


MACROS = {
    "i": {"EXVIiBeta": sx("i", "beta"), "EXVIiImbalance": sx("i", "imbalance"),
          "EXVIiWeight": sx("i", "weight_program")},
    "ii": {"EXVIiiInf": sx("ii", "inf_not_attained"), "EXVIiiValue": sx("ii", "value"),
           "EXVIiiWOne": sx("ii", "w1"), "EXVIiiWTwo": sx("ii", "w2")},
    "iii": {"EXVIiiiWeightMinus": sx("iii", "sq_w(z=-1)"), "EXVIiiiWeightZero": sx("iii", "sq_w(z=0)"),
            "EXVIiiiWeightPlus": sx("iii", "sq_w(z=1)"), "EXVIiiiEstimand": sx("iii", "sq_estimand"),
            "EXVIiiiBias": sx("iii", "sq_bias")},
    "iv": {"EXVIivWeightMinus": sx("iv", "ukl_w(z=-1)"), "EXVIivWeightZero": sx("iv", "ukl_w(z=0)"),
           "EXVIivWeightPlus": sx("iv", "ukl_w(z=1)"), "EXVIivEstimand": sx("iv", "ukl_estimand"),
           "EXVIivBias": sx("iv", "ukl_bias")},
    "v": {"EXVIvBetaOne": sx("v", "beta1"), "EXVIvBetaTwo": sx("v", "beta2"),
          "EXVIvAlphaZero": sx("v", "alpha(x=0)"), "EXVIvAlphaOne": sx("v", "alpha(x=1)"),
          "EXVIvAlphaTwo": sx("v", "alpha(x=2)"), "EXVIvNormSq": sx("v", "norm_sq"),
          "EXVIvMean": sx("v", "mean_e")},
}  # fmt: skip
macros = {k: v for ex, ms in MACROS.items() if verdict[ex] for k, v in ms.items()}
macros["EXVINumberAgreeing"] = str(sum(verdict.values()))
rec.write_macros(macros)
rec.finalize(R=None, n=None, warnings=len(warning_messages), failures=int(sum(not v for v in verdict.values())))

PosixPath('/Users/mkato/Dropbox/研究/generalizedrieszregression/genriesz/notebooks/experiments/results/E-16/stage0/kernel.json')